# Faruq-v3 — Geometry Family Factorization Finalize

Jalankan setelah notebook GEO-SHARED60 dan GEO-FAM35x3 selesai. Notebook ini tidak training, tidak membutuhkan GPU, dan tidak membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
BASE=(
 'experiments/faruq-v3-geometry-conditioning-paired-confirmation-v1/val_reports/geometry_conditioning_paired_three_seed_confirmation.json',
 'experiments/faruq-v3-geometry-family-effect-decomposition-v1/geometry_family_effect_decomposition.json',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/experiment_manifest.json',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/experiment_manifest.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed123_val.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/experiment_manifest.json',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed2026_val.json')
REPORTS=tuple(f'experiments/faruq-v3-geometry-family-factorization-v1/val_reports/{arm}_seed{seed}_val.json' for seed in (42,123,2026) for arm in ('GEO-SHARED60','GEO-FAM35x3'))
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=BASE+REPORTS)
CONFIRM=require_project_artifact(PROJECT_ROOT,BASE[0]); DECOMP=require_project_artifact(PROJECT_ROOT,BASE[1])
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-geometry-family-factorization-v1'
print('PROJECT:',PROJECT_ROOT); print('OUTPUT:',OUTPUT_ROOT)

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_geometry_family_factorization','--data-root','/content/faruq-development-v3-grouped','--project-root',str(PROJECT_ROOT),'--confirmation-summary',str(CONFIRM),'--family-decomposition',str(DECOMP),'--output-root',str(OUTPUT_ROOT),'--stage','finalize']
print('FINALIZE:', ' '.join(command)); subprocess.run(command,cwd=REPO,check=True)

In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'val_reports/geometry_family_factorization_three_seed.json'
result=json.loads(SUMMARY.read_text(encoding='utf-8')); assert result['test_opened'] is False
rows=[]
for seed in result['seeds']:
    for arm in ('GEO-SHARED60','GEO-FAM35x3'):
        values=result['per_seed'][str(seed)]['results'][arm]
        rows.append({'seed':seed,'model':arm,'Macro':values['macro_map50_95'],'Bottom3':values['bottom3_class_map50_95'],'Worst':values['worst_class_map50_95'],'SizeMean':values['size_class_mean_map50_95']})
display(pd.DataFrame(rows).style.format({'Macro':'{:.2%}','Bottom3':'{:.2%}','Worst':'{:.2%}','SizeMean':'{:.2%}'}))
delta=[{'metric':metric,'mean_delta':values['delta_mean'],'improved_seeds':values['improved_seeds'],'min_delta':values['delta_min'],'max_delta':values['delta_max']} for metric,values in result['aggregate'].items()]
display(pd.DataFrame(delta).style.format({'mean_delta':'{:+.2%}','min_delta':'{:+.2%}','max_delta':'{:+.2%}'}))
print('SCIENTIFIC STATUS:',result['scientific_status']); print('DECISION:',result['decision']); print('NEXT:',result['next_action']); print('SUMMARY:',SUMMARY)
print('Test tetap terkunci.')